In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

In [3]:
root_path = (Path(os.getcwd()).parent.parent / "src").as_posix()
root_path

'/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework/src'

In [4]:
sys.path.append(root_path)

In [5]:
from wilsonelser.transformation.data_transformer import DataTransformer
from wilsonelser.transformation.base_data_transformer import BaseDataTransformer
from wilsonelser.transformation.table_writer import TableWriter
from wilsonelser.transformation.table_reader import TableReader


In [6]:
config_file = "use_cases/p2c_human_nutrition_open_orders/human_nutrition_open_orders.yml"
table_reader = TableReader(config_file)
transformer = BaseDataTransformer(config_file)
transformer2 = DataTransformer(config_file)
writer = TableWriter(config_file)


In [7]:
df_ibm_joined = transformer2.apply_joins("combine_ibmdb2")

INFO:adm.integration.base_integration:Applying joins for combine_id: combine_ibmdb2
INFO:adm.integration.base_integration:Reading and selecting columns for source_id: ibmdb2_opinprod_product with alias: op
INFO:adm.integration.base_integration:Joining iod with op using condition: iod.inor_id = op.prod_inor_id and join type: inner
INFO:adm.integration.base_integration:Schema after joining iod with op: StructType([StructField('inor_id', IntegerType(), False), StructField('inor_copc_id', IntegerType(), False), StructField('inor_curr_id', IntegerType(), False), StructField('inor_tc_terms_code', StringType(), False), StructField('inor_soldto', StringType(), False), StructField('inor_ship_date', DateType(), True), StructField('inor_status_code', StringType(), False), StructField('inor_order_number', StringType(), False), StructField('inor_order_date', DateType(), False), StructField('prod_product_id', IntegerType(), False), StructField('prod_inor_id', IntegerType(), False), StructField('prod

In [8]:
df_ibm_exch = transformer.apply_transformations("ibm_exchange_rate", df_ibm_joined)

INFO:adm.integration.base_integration:Starting transformations for ID: ibm_exchange_rate
INFO:adm.integration.base_integration:Applying function Name: use_cases.custom_transformations.hn_tx.with_exchange_rate


In [9]:
df_ibm_output = transformer.apply_transformations("ibm_output", df_ibm_exch)

INFO:adm.integration.base_integration:Starting transformations for ID: ibm_output


In [10]:
df_jde_joined = transformer2.apply_joins("combine_jde")

INFO:adm.integration.base_integration:Applying joins for combine_id: combine_jde
INFO:adm.integration.base_integration:Reading and selecting columns for source_id: jdexeeu_sales_order_line with alias: b
INFO:adm.integration.base_integration:Joining a with b using condition: a.shkcoo = b.sdkcoo and a.shdoco = b.sddoco and a.shdcto = b.sddcto and join type: inner
INFO:adm.integration.base_integration:Schema after joining a with b: StructType([StructField('shkcoo', StringType(), False), StructField('shdoco', DoubleType(), False), StructField('shdcto', StringType(), False), StructField('shtrdj', DecimalType(18,0), True), StructField('sdkcoo', StringType(), False), StructField('sddoco', DoubleType(), False), StructField('sddcto', StringType(), False), StructField('sdan8', DoubleType(), True), StructField('sdlnid', DoubleType(), False), StructField('sditm', DoubleType(), True), StructField('sdlitm', StringType(), True), StructField('sddsc1', StringType(), True), StructField('sdsoqs', DoubleT

In [11]:
df_jde_output = transformer.apply_transformations("jde_output", df_jde_joined)

INFO:adm.integration.base_integration:Starting transformations for ID: jde_output


In [12]:
df_out = df_ibm_output.union(df_jde_output)

In [13]:
spark.conf.set("spark.sql.legacy.charVarcharAsString", True)

In [14]:
writer.write_table("hn_open_orders", df_out)

INFO:adm.integration.base_integration:Number of records written to table sandbox.de_ref_silver_mikesturwoldcontractor.hn_open_orders: 22781


In [16]:
display(spark.table("sandbox.de_ref_silver_mikesturwoldcontractor.hn_open_orders").groupBy("source").count())

,source,count
0,JDE EDU,19256
1,IBM,3525
